# [v4] Train lại model tư thế từ bộ video 2 cam (quay 11/2026)

Pipeline giữ nguyên v3 (cam hông 90° → MediaPipe → góc cổ / thân / cúi đầu → **RULA nhóm B làm teacher** → Transformer → ONNX),
đổi những điểm sau để khớp bộ quay mới bằng `record_session.py`:

| | v3 | v4 (notebook này) |
|---|---|---|
| Dữ liệu | thư mục `Video_TrainAI_KhoaLuan`, 1 file / clip | `video_tach/`: `P0x_S01_<CLIP>_SIDE.mp4` + `_FRONT.mp4` quay đồng thời |
| Baseline | toàn bộ clip CALIBRATION (dính cả lúc vỗ tay) | **15 s giữa CALIBRATION** (giây 3–18), giống công cụ quay |
| Khung đưa vào train | mọi khung | bỏ **3 s đầu/cuối** mỗi clip và 3 s sau mỗi lần đổi mức (TRANSITION) — đoạn người đang chuyển tư thế |
| Đầu ra model | 1 logit GOOD/BAD | **logit GOOD/BAD (giữ nguyên, app không phải sửa)** + đầu phụ 6 lớp: GOOD, FORWARD_HEAD, LEAN_FORWARD, SLOUCH, LEAN_BACKWARD, NECK_EXTENSION |
| Cam trước | chỉ vẽ biểu đồ lệch vai | đo **lệch vai + nhún vai** trên clip E1/E2/E9–E11, gợi ý ngưỡng (không train) |

Đầu vào model vẫn là `x (batch, 20, 6)` với 6 đặc trưng cũ, nên `engine.py` chạy được file ONNX mới mà **không sửa code**
(`sess.run(None, ...)[0]` vẫn là logit BAD). Đầu 6 lớp nằm ở output thứ hai `posture_logits`, app dùng sau nếu muốn.

**Cách chạy**
1. Nén thư mục `D:\KLTN\video_tach` thành ZIP (**bỏ thư mục `_cu`** – bản quay hỏng). Tạo **Kaggle Dataset riêng tư** (Private) từ file ZIP đó.
2. Notebook → **Add Input** dataset đó. Settings → **Internet: On**. Accelerator: không cần (GPU chỉ nhanh hơn phần train một chút).
3. **Run All**. Notebook tự tìm thư mục chứa `*_SIDE.mp4` trong `/kaggle/input` (sửa `DATA_DIR` ở Phần 1 nếu muốn chỉ định).
4. Tải `posture_model_v4.zip` ở tab Output.
5. (Tuỳ chọn) Lần chạy sau: thêm Output của lần này làm Input → notebook dùng lại landmark `.npz`, khỏi trích lại MediaPipe.
6. (Tuỳ chọn) Thêm dataset chứa `posture_transformer.onnx` v3 đang chạy trong app → Phần 6 so sánh v3 với v4 trên dữ liệu mới.

## Phần 0 — Cài đặt

In [ ]:
import subprocess, sys, importlib.util
need = [p for p, m in [('mediapipe', 'mediapipe'), ('onnx', 'onnx'), ('onnxruntime', 'onnxruntime'), ('onnxscript', 'onnxscript')]
        if importlib.util.find_spec(m) is None]
if need:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-1500:])
import mediapipe as mp, torch, numpy as np, pandas as pd, cv2, onnx, onnxruntime, sklearn
print('mediapipe', mp.__version__, '| torch', torch.__version__, '| onnxruntime', onnxruntime.__version__,
      '| numpy', np.__version__, '| GPU:', torch.cuda.is_available())

## Phần 1 — Cấu hình + tìm video

Kịch bản clip lấy từ `posture_model/recording/script_v2.json` (bản dùng khi quay). Chỉ cần thời lượng và mốc đổi mức để bỏ đoạn chuyển tư thế.

In [ ]:
import os, re, glob, json, time, shutil
from pathlib import Path

DATA_DIR = os.environ.get('KLTN_DATA_DIR')     # None = tự tìm trong /kaggle/input; hoặc ghi đường dẫn, vd '/kaggle/input/kltn-video-tach/video_tach'
INPUT_ROOT = os.environ.get('KLTN_INPUT_ROOT', '/kaggle/input')
WORK = os.environ.get('KLTN_WORK', '/kaggle/working')
EXCLUDE_PERSONS = {'P99'}                       # P99 = buổi quay thử
TARGET_FPS = 10
MARGIN = 3.0                                    # giây bỏ ở đầu/cuối clip và sau mỗi mốc đổi mức (= margin_sec của công cụ quay)
CAL_START, CAL_SEC = 3.0, 15.0                  # baseline = 15 s giữa clip CALIBRATION
SEED = 0

# clip -> (thời lượng s, [giây bắt đầu từng mức]) theo script_v2.json
SCRIPT = {
    'CALIBRATION': (20, [0]), 'GOOD_01': (60, [0]), 'GOOD_02': (60, [0]), 'NATURAL_01': (120, [0]),
    'FORWARD_HEAD_L1': (45, [0]), 'FORWARD_HEAD_L2': (45, [0]), 'LEAN_FORWARD_L1': (45, [0]), 'LEAN_FORWARD_L2': (45, [0]),
    'SLOUCH_L1': (45, [0]), 'SLOUCH_L2': (45, [0]), 'LEAN_BACKWARD_L1': (45, [0]), 'LEAN_BACKWARD_L2': (45, [0]),
    'NECK_EXTENSION': (45, [0]), 'CHIN_ON_HAND_L': (30, [0]), 'CHIN_ON_HAND_R': (30, [0]),
    'TRANSITION_01': (120, [0, 15, 30, 45, 60, 75, 90, 105]),
    'TRANSIENT_DRINK': (20, [0]), 'TRANSIENT_PHONE': (20, [0]), 'TRANSIENT_PICKUP': (20, [0]),
    'TRANSIENT_STRETCH': (20, [0]), 'TRANSIENT_TURN_HEAD': (20, [0]), 'TRANSIENT_LOOK_KEYBOARD': (20, [0]),
    'SHOULDER_TILT_L': (30, [0, 15]), 'SHOULDER_TILT_R': (30, [0, 15]),
    'SHOULDER_RAISE_L': (30, [0, 15]), 'SHOULDER_RAISE_R': (30, [0, 15]), 'SHOULDER_SHRUG_BOTH': (30, [0, 15]),
}
# Cam hông: clip dùng train (khối A trừ CALIBRATION, B, C, D). Khối E (cam trước) và AWAY không train.
TRAIN_CLIPS = [c for c in SCRIPT if c != 'CALIBRATION' and not c.startswith('SHOULDER_')]
# Cam trước: clip dùng để đo lệch vai / nhún vai (không train)
FRONT_CLIPS = ['CALIBRATION', 'GOOD_01', 'GOOD_02', 'SHOULDER_TILT_L', 'SHOULDER_TILT_R',
               'SHOULDER_RAISE_L', 'SHOULDER_RAISE_R', 'SHOULDER_SHRUG_BOTH']
NAME_RE = re.compile(r'^(P\d+)_(S\d+)_(.+)_(SIDE|FRONT)$')

def find_videos(root):
    out = {}
    for p in sorted(Path(root).rglob('*.mp4')):
        if '_cu' in p.parts or '__cu' in p.stem:         # bản quay hỏng đã cất đi
            continue
        m = NAME_RE.match(p.stem)
        if not m or m.group(1) in EXCLUDE_PERSONS:
            continue
        if p.stem in out:
            print('Trùng tên, bỏ bản sau:', p); continue
        out[p.stem] = str(p)
    return out

if DATA_DIR is None:
    hits = sorted({str(p.parent) for p in Path(INPUT_ROOT).rglob('*_SIDE.mp4') if '_cu' not in p.parts})
    if not hits:
        raise FileNotFoundError(f'Không thấy *_SIDE.mp4 trong {INPUT_ROOT} — Add Input dataset video hoặc sửa DATA_DIR')
    DATA_DIR = os.path.commonpath(hits)
VIDEOS = find_videos(DATA_DIR)
LM_DIR = f'{WORK}/lm'; OUT_DIR = f'{WORK}/posture_model'
os.makedirs(LM_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK)

def parse(stem):
    p, s, clip, cam = NAME_RE.match(stem).groups()
    return p, s, clip, cam

jobs = [(st, v) for st, v in VIDEOS.items()
        if (parse(st)[3] == 'SIDE' and parse(st)[2] in TRAIN_CLIPS + ['CALIBRATION'])
        or (parse(st)[3] == 'FRONT' and parse(st)[2] in FRONT_CLIPS)]
people = sorted({parse(s)[0] for s, _ in jobs})
print('DATA_DIR =', DATA_DIR)
print(f'{len(VIDEOS)} video hợp lệ, cần trích {len(jobs)} video; người: {people}')
tab = pd.DataFrame([dict(person=parse(s)[0] + '_' + parse(s)[1], cam=parse(s)[3], clip=parse(s)[2]) for s, _ in jobs])
print(tab.pivot_table(index='person', columns='cam', values='clip', aggfunc='count').to_string())
miss = [(p, c) for p in sorted(tab.person.unique()) for c in TRAIN_CLIPS + ['CALIBRATION']
        if not ((tab.person == p) & (tab.cam == 'SIDE') & (tab['clip'] == c)).any()]
print('Thiếu clip cam hông:', miss or 'không')

## Phần 2 — Module đặc trưng (chép nguyên `posture_model/posture_features.py` của app)

Train và app phải tính góc giống hệt nhau, nên ô này là bản sao byte-đúng của file trong repo (chỉ đổi xuống dòng CRLF → LF).
Nếu sửa file trong repo thì chép lại vào đây.

In [ ]:
%%writefile posture_features.py
"""
Đặc trưng tư thế + RULA nhóm B + công thức lệch vai — DÙNG CHUNG cho lúc train và lúc chạy realtime.

- Cam hông 90° (train + chạy thật): góc cổ / thân / cúi đầu -> RULA nhóm B (teacher) -> Transformer (student).
- Cam laptop chính diện (chỉ lúc chạy thật): công thức lệch vai, KHÔNG train.

Đầu vào mỗi khung: 33 landmark MediaPipe Pose 2D (toạ độ pixel x, y, visibility),
world landmark 3D chỉ dùng để chọn phía cơ thể gần camera.
"""
import math
import numpy as np

# chỉ số MediaPipe Pose: (trái, phải)
IDX = {"ear": (7, 8), "eye": (2, 5), "shoulder": (11, 12), "hip": (23, 24)}
NOSE = 0

FEATURES = ["neck", "trunk", "head_pitch", "vis_ear", "vis_shoulder", "vis_hip"]
CALIBRATED = ["neck", "trunk", "head_pitch"]      # trừ giá trị lúc ngồi chuẩn của từng người
ANGLE_SCALE = 30.0                                 # chia góc cho 30° trước khi vào model

# ================================================================ CAM HÔNG 90°
def choose_side(lm_seq, world_seq=None):
    """0 = trái, 1 = phải. Ưu tiên world z (nhỏ = gần camera), dự phòng bằng visibility."""
    if world_seq is not None and np.isfinite(world_seq).any():
        zl = np.nanmedian(world_seq[:, 11, 2] + world_seq[:, 23, 2])
        zr = np.nanmedian(world_seq[:, 12, 2] + world_seq[:, 24, 2])
        if np.isfinite(zl) and np.isfinite(zr):
            return 0 if zl < zr else 1
    vl = np.nanmean(lm_seq[:, [7, 11, 23], 2]); vr = np.nanmean(lm_seq[:, [8, 12, 24], 2])
    return 0 if vl >= vr else 1


def facing_sign(lm_seq, side):
    """+1 nếu người quay mặt sang phải khung hình, -1 nếu sang trái (mũi so với tai)."""
    d = np.nanmedian(lm_seq[:, NOSE, 0] - lm_seq[:, IDX["ear"][side], 0])
    return 1.0 if (np.isfinite(d) and d > 0) else -1.0


def _ang(v, f):
    """Góc (độ) của vector v so với phương thẳng đứng hướng LÊN, dương khi nghiêng về phía mặt."""
    return math.degrees(math.atan2(v[0] * f, -v[1]))


def frame_angles(p, side, f):
    """p: (33,3) pixel x,y,vis. Trả dict góc (độ) + visibility, None nếu thiếu tai/vai/hông."""
    g = lambda k: p[IDX[k][side], :2]
    ear, eye, sh, hip = g("ear"), g("eye"), g("shoulder"), g("hip")
    if not np.isfinite(np.concatenate([ear, sh, hip])).all():
        return None
    trunk = _ang(sh - hip, f)                       # thân so với phương đứng (+ = gập ra trước, − = ngả sau)
    neck = _ang(ear - sh, f) - trunk                # cổ so với đường thân (+ = gập)
    # cúi đầu: đường tai -> mắt so với phương ngang (+ = mắt thấp hơn tai = cúi)
    head_pitch = math.degrees(math.atan2(eye[1] - ear[1], (eye[0] - ear[0]) * f)) if np.isfinite(eye).all() else np.nan
    vis = lambda k: float(p[IDX[k][side], 2])
    return dict(neck=neck, trunk=trunk, head_pitch=head_pitch,
                vis_ear=vis("ear"), vis_shoulder=vis("shoulder"), vis_hip=vis("hip"))


def sequence_angles(lm_seq, side, f):
    """(T,33,3) -> (T, len(FEATURES)) góc thô (chưa hiệu chỉnh), NaN nếu thiếu."""
    out = np.full((len(lm_seq), len(FEATURES)), np.nan, np.float32)
    for t, p in enumerate(lm_seq):
        if not np.isfinite(p).any():
            continue
        a = frame_angles(p, side, f)
        if a is not None:
            out[t] = [a[k] for k in FEATURES]
    return out


def calibration_baseline(raw):
    """raw: (T,F) góc thô lúc ngồi chuẩn -> dict baseline cho các góc cần hiệu chỉnh."""
    return {k: float(np.nanmedian(raw[:, FEATURES.index(k)])) for k in CALIBRATED}


def apply_calibration(raw, base):
    x = raw.copy()
    for k in CALIBRATED:
        x[:, FEATURES.index(k)] -= base[k]
    return x


def to_model_input(cal):
    """Góc đã hiệu chỉnh -> mảng chuẩn hoá cho model (NaN -> 0, visibility giữ nguyên 0..1)."""
    x = cal.copy()
    for i, k in enumerate(FEATURES):
        if not k.startswith("vis_"):
            x[:, i] = x[:, i] / ANGLE_SCALE
    return np.nan_to_num(x, nan=0.0).astype(np.float32)

# ================================================================ RULA NHÓM B (teacher)
# McAtamney & Corlett (1993). Chỉ Cổ – Thân – Chân; bỏ nhóm A, điểm cơ, điểm lực.
# Side cam không thấy xoay / nghiêng sang bên -> các cờ điều chỉnh = 0. Chân mặc định = 1.
TABLE_B = {  # neck -> [T1L1, T1L2, T2L1, T2L2, ..., T6L1, T6L2]
    1: [1, 3, 2, 3, 3, 4, 5, 5, 6, 6, 7, 7], 2: [2, 3, 2, 3, 4, 5, 5, 5, 6, 7, 7, 7],
    3: [3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7], 4: [5, 5, 5, 6, 6, 7, 7, 7, 7, 7, 8, 8],
    5: [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], 6: [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9],
}
BAD_SCORE = 3          # điểm nhóm B >= 3 -> BAD
NEUTRAL_TOL = 5.0      # thân: <= 5° so với lúc hiệu chỉnh vẫn coi là thẳng (1 điểm)
NECK_EXT_TOL = 15.0    # cổ chỉ tính "ngửa" (4 điểm) khi ngửa hơn lúc hiệu chỉnh > 15°
LEAN_BACK_TOL = 15.0   # luật bổ sung ngoài RULA: thân ngả sau > 15° -> BAD (LEAN_BACKWARD)
HEAD_FROM_PITCH = True # góc cổ = max(tai–vai so với thân, cúi mặt tai->mắt): bắt kiểu gù mà cúi mặt


def score_neck(a):
    if a < -NECK_EXT_TOL: return 4
    if a <= 10:           return 1
    if a <= 20:           return 2
    return 3


def score_trunk(a):
    if a <= NEUTRAL_TOL: return 1          # thẳng, hoặc ngả sau có tựa
    if a <= 20:          return 2
    if a <= 60:          return 3
    return 4


def group_b_score(neck, trunk, legs=1):
    """Góc cổ, thân (độ, đã hiệu chỉnh) -> điểm nhóm B (1..9), NaN nếu thiếu."""
    if not (np.isfinite(neck) and np.isfinite(trunk)):
        return np.nan
    return TABLE_B[score_neck(neck)][(score_trunk(trunk) - 1) * 2 + (legs - 1)]


def b_label(neck, trunk):
    """-> 0 = GOOD, 1 = BAD, NaN nếu thiếu."""
    b = group_b_score(neck, trunk)
    if not np.isfinite(b):
        return np.nan
    return int(b >= BAD_SCORE or trunk < -LEAN_BACK_TOL)


def rula_frame_label(cal_row, margin=2.0):
    """cal_row: 1 khung đặc trưng đã hiệu chỉnh. Trả (nhãn 0/1/NaN, ổn_định?).
    'Ổn định' = nhãn không đổi khi xê dịch cổ/thân ±margin độ (khung sát ngưỡng sẽ không ổn định)."""
    n, t, hp = (cal_row[FEATURES.index(k)] for k in ("neck", "trunk", "head_pitch"))
    if HEAD_FROM_PITCH and np.isfinite(hp) and np.isfinite(n):
        n = max(n, hp)
    lab = b_label(n, t)
    if not np.isfinite(lab):
        return np.nan, False
    stable = all(b_label(n + dn, t + dt) == lab for dn in (-margin, margin) for dt in (-margin, margin))
    return int(lab), stable

# ================================================================ CAM LAPTOP: LỆCH VAI (không train)
SHOULDER_TILT_ON = 3.5     # |lệch vai so với lúc hiệu chỉnh| > 3.5° -> lệch (clip P01 nghiêng chủ ý chỉ ~4–5°)
SHOULDER_TILT_OFF = 2.5    # < 2.5° mới coi là hết lệch (trễ, tránh bật/tắt liên tục); ngồi thẳng dao động ~±2°
MIN_SHOULDER_VIS = 0.5     # vai bị che / ra khỏi khung -> bỏ khung


def shoulder_tilt(p):
    """p: (33,3) pixel x,y,vis từ cam CHÍNH DIỆN. Góc (độ) của đường nối 2 vai so với phương ngang.
    Đường vai lấy từ vai bên trái ảnh -> vai bên phải ảnh, nên ảnh lật gương (selfie) vẫn đúng độ lớn.
    Dấu: + = vai bên phải ẢNH thấp hơn. Chỉ dùng |tilt − baseline| để cảnh báo."""
    L, R = p[11], p[12]
    if not (np.isfinite(L).all() and np.isfinite(R).all()) or min(L[2], R[2]) < MIN_SHOULDER_VIS:
        return np.nan
    dx, dy = R[0] - L[0], R[1] - L[1]
    if dx < 0:
        dx, dy = -dx, -dy
    if dx < 1e-6:
        return np.nan
    return math.degrees(math.atan2(dy, dx))


MIN_SHOULDER_WIDTH_PX = 20.0   # hai vai gần nhau hơn mức này (xoay người / ra xa) -> tỉ lệ vô nghĩa


def shoulder_raise(p):
    """p: (33,3) pixel x,y,vis từ cam CHÍNH DIỆN. Độ nâng vai (đo nhún vai) = khoảng cách DỌC tai -> vai (trung bình hai bên)
    chia độ rộng vai |x vai trái − x vai phải| (để không phụ thuộc khoảng cách tới cam). Vai nâng lên -> giá trị GIẢM.

    Nguồn: Estrada, Vea, Devaraj (2023), Applied Sciences 13(5402): đặc trưng khoảng cách đầu -> vai (cơ thang) cho mô hình vai
    "levelled (not raised or rounded)". Dùng TAI (7, 8) thay mũi vì mũi dịch theo hướng xoay mặt; thiếu một tai thì dùng tai
    còn lại. Chỉ so với baseline lúc hiệu chỉnh. Cúi đầu / đưa đầu ra trước cũng làm giá trị giảm -> engine phải loại bằng
    nhánh cổ/thân (cam hông). Trả nan nếu vai/tai không đủ tin cậy hoặc độ rộng vai quá nhỏ."""
    L, R = p[11], p[12]
    if not (np.isfinite(L).all() and np.isfinite(R).all()) or min(L[2], R[2]) < MIN_SHOULDER_VIS:
        return np.nan
    width = abs(float(R[0] - L[0]))
    if width < MIN_SHOULDER_WIDTH_PX:
        return np.nan
    ds = []
    for ear, sh in ((p[7], L), (p[8], R)):
        if np.isfinite(ear).all() and ear[2] >= MIN_SHOULDER_VIS:
            ds.append(float(sh[1] - ear[1]))          # ảnh: y tăng xuống dưới -> vai ở dưới tai = dương
    if not ds:
        return np.nan
    return float(np.mean(ds)) / width


def sequence_shoulder_tilt(lm_seq):
    return np.array([shoulder_tilt(p) if np.isfinite(p).any() else np.nan for p in lm_seq], np.float32)

## Phần 3 — Trích landmark MediaPipe (~10 fps, ảnh ½, chạy song song, có cache)

Cache tìm theo thứ tự: `/kaggle/working/lm` → mọi thư mục `lm/` có sẵn trong `/kaggle/input` (Output lần trước).

In [ ]:
import urllib.request
from concurrent.futures import ProcessPoolExecutor

TASK = next((str(p) for p in Path(INPUT_ROOT).rglob('pose_landmarker_full.task')), None) if Path(INPUT_ROOT).exists() else None
if TASK is None:
    TASK = f'{WORK}/pose_landmarker_full.task'
    if not os.path.exists(TASK):
        urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
                                   'pose_landmarker_full/float16/latest/pose_landmarker_full.task', TASK)
print('Model pose:', TASK)

CACHE_DIRS = [LM_DIR] + (sorted({str(p.parent) for p in Path(INPUT_ROOT).rglob('lm/*.npz')}) if Path(INPUT_ROOT).exists() else [])

def cached(stem):
    for d in CACHE_DIRS:
        f = f'{d}/{stem}.npz'
        if os.path.exists(f):
            return f
    return None

def extract(job):
    stem, v = job
    import cv2, numpy as np, mediapipe as mp
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    out = f'{LM_DIR}/{stem}.npz'
    det = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=TASK), running_mode=vision.RunningMode.VIDEO, num_poses=1))
    cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30; step = max(1, round(fps / TARGET_FPS))
    L, W, T = [], [], []; i = -1; w = h = 0
    while cap.grab():
        i += 1
        if i % step: continue
        ok, fr = cap.retrieve()
        if not ok: break
        h, w = fr.shape[:2]
        sm = cv2.resize(fr, (w // 2, h // 2))
        r = det.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(sm, cv2.COLOR_BGR2RGB)),
                                 int(i * 1000 / fps))
        if r.pose_landmarks:
            L.append([[p.x * w, p.y * h, p.visibility] for p in r.pose_landmarks[0]])
            W.append([[p.x, p.y, p.z] for p in r.pose_world_landmarks[0]])
        else:
            L.append(np.full((33, 3), np.nan)); W.append(np.full((33, 3), np.nan))
        T.append(i / fps)
    cap.release(); det.close()
    np.savez_compressed(out, lm=np.array(L, np.float32).reshape(-1, 33, 3), world=np.array(W, np.float32).reshape(-1, 33, 3),
                        t=np.array(T), fps=fps, wh=np.array([w, h]))
    return stem, len(T)

LM = {s: cached(s) for s, _ in jobs}
todo = [j for j in jobs if LM[j[0]] is None]
print(f'{len(jobs) - len(todo)} video có cache, cần trích {len(todo)} video')
t0 = time.time()
if todo:
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        for k, (stem, n) in enumerate(ex.map(extract, todo), 1):
            LM[stem] = f'{LM_DIR}/{stem}.npz'
            print(f'[{k}/{len(todo)}] {stem}: {n} khung  ({time.time() - t0:.0f}s)', flush=True)
for s, f in LM.items():                         # gom cache cũ vào Output để lần sau dùng lại
    if f and not f.startswith(LM_DIR):
        shutil.copy(f, f'{LM_DIR}/{s}.npz'); LM[s] = f'{LM_DIR}/{s}.npz'

## Phần 4 — Teacher RULA nhóm B + nhãn 6 lớp + cắt cửa sổ (cam hông)

- Hiệu chỉnh theo **người + phiên + độ phân giải**: baseline = trung vị góc ở giây 3–18 của `CALIBRATION` (thiếu thì 10 s đầu `GOOD_01` sau 3 s).
- Nhãn GOOD/BAD mỗi khung = `rula_frame_label` của app (Table B, ngưỡng 3, luật ngả sau > 15°), giữ nguyên v3.
- Nhãn lớp phụ suy từ **cùng góc mà teacher dùng** (cổ = max(cổ, cúi mặt), thân), thứ tự ưu tiên:
  GOOD nếu teacher GOOD → LEAN_BACKWARD (thân < −15°) → SLOUCH (thân > 20° **và** cổ > 20°) → LEAN_FORWARD (thân > 20°)
  → FORWARD_HEAD (cổ > 20°) → NECK_EXTENSION (cổ < −15°). Các mốc 20° và −15° trùng ngưỡng kiểm "đạt" lúc quay.
- Bỏ khung: 3 s đầu/cuối clip, 3 s sau mỗi mốc đổi mức.
- Cửa sổ 2 s (20 khung), bước 0,5 s. Bỏ cửa sổ: thiếu điểm, hông khó thấy, nhãn lẫn (BAD 20–80%), > 40% khung sát ngưỡng ±2° (như v3).
  Lớp phụ của cửa sổ = lớp chiếm ≥ 70% khung; không đủ thì lớp = −1 (không tính vào loss lớp, vẫn train GOOD/BAD).

In [ ]:
import sys, importlib
sys.path.insert(0, WORK)
import posture_features; importlib.reload(posture_features)
from posture_features import (FEATURES, choose_side, facing_sign, sequence_angles, calibration_baseline,
                              apply_calibration, to_model_input, rula_frame_label, NECK_EXT_TOL, LEAN_BACK_TOL, HEAD_FROM_PITCH)

CLASSES = ['GOOD', 'FORWARD_HEAD', 'LEAN_FORWARD', 'SLOUCH', 'LEAN_BACKWARD', 'NECK_EXTENSION']
FLEX = 20.0                    # cổ / thân gập > 20° (mốc RULA điểm 3 và mốc kiểm "đạt" khi quay)
WIN, STRIDE = 20, 5
MIN_PURE, MIN_STABLE, MIN_VALID, MIN_VIS_HIP = 0.8, 0.6, 0.8, 0.3
CLS_MIN_PURE = 0.7
# lớp "dự kiến" theo kịch bản quay — chỉ để kiểm teacher, KHÔNG dùng làm nhãn
INTENT = {'GOOD_01': 'GOOD', 'GOOD_02': 'GOOD', 'FORWARD_HEAD_L1': 'FORWARD_HEAD', 'FORWARD_HEAD_L2': 'FORWARD_HEAD',
          'LEAN_FORWARD_L1': 'LEAN_FORWARD', 'LEAN_FORWARD_L2': 'LEAN_FORWARD', 'SLOUCH_L1': 'SLOUCH', 'SLOUCH_L2': 'SLOUCH',
          'LEAN_BACKWARD_L1': 'LEAN_BACKWARD', 'LEAN_BACKWARD_L2': 'LEAN_BACKWARD', 'NECK_EXTENSION': 'NECK_EXTENSION'}
iN, iT, iH, iV = (FEATURES.index(k) for k in ('neck', 'trunk', 'head_pitch', 'vis_hip'))

def frame_class(row, lab):
    if not np.isfinite(lab):
        return -1
    if lab == 0:
        return 0
    n, t, hp = row[iN], row[iT], row[iH]
    if HEAD_FROM_PITCH and np.isfinite(hp):
        n = max(n, hp)
    if t < -LEAN_BACK_TOL:          return CLASSES.index('LEAN_BACKWARD')
    if t > FLEX and n > FLEX:       return CLASSES.index('SLOUCH')
    if t > FLEX:                    return CLASSES.index('LEAN_FORWARD')
    if n > FLEX:                    return CLASSES.index('FORWARD_HEAD')
    if n < -NECK_EXT_TOL:           return CLASSES.index('NECK_EXTENSION')
    return -1                       # BAD nhưng không rơi vào lớp nào (hiếm)

def judged_mask(t, clip):
    dur, starts = SCRIPT[clip]
    end = min(dur, t[-1] + 1e-6) if len(t) else dur
    m = (t >= MARGIN) & (t < end - MARGIN)
    for s in starts[1:]:
        m &= ~((t >= s) & (t < s + MARGIN))
    return m

side_lm = [(s, f) for s, f in LM.items() if parse(s)[3] == 'SIDE']
meta = []
for s, f in side_lm:
    p, ses, clip, _ = parse(s)
    z = np.load(f)
    meta.append(dict(file=f, stem=s, person=p, session=ses, clip=clip, setup='x'.join(str(int(v)) for v in z['wh'])))
meta = pd.DataFrame(meta)

X, Y, C, G, INFO, frames_rows, BASES = [], [], [], [], [], [], {}
for (person, session, setup), grp in meta.groupby(['person', 'session', 'setup']):
    data = {r.clip: np.load(r.file) for r in grp.itertuples()}
    all_lm = np.concatenate([d['lm'] for d in data.values()]); all_w = np.concatenate([d['world'] for d in data.values()])
    side = choose_side(all_lm, all_w); f = facing_sign(all_lm, side)
    raw = {c: sequence_angles(d['lm'], side, f) for c, d in data.items()}
    tt = {c: d['t'] - d['t'][0] for c, d in data.items()}
    if 'CALIBRATION' in raw:
        sel = (tt['CALIBRATION'] >= CAL_START) & (tt['CALIBRATION'] < CAL_START + CAL_SEC)
        base = calibration_baseline(raw['CALIBRATION'][sel]); src = 'CALIBRATION 3–18 s'
    elif 'GOOD_01' in raw:
        sel = (tt['GOOD_01'] >= MARGIN) & (tt['GOOD_01'] < MARGIN + 10)
        base = calibration_baseline(raw['GOOD_01'][sel]); src = 'GOOD_01 10 s'
    else:
        print(f'{person}_{session} [{setup}]: không có CALIBRATION/GOOD_01 -> bỏ'); continue
    BASES[f'{person}_{session}_{setup}'] = dict(side=int(side), facing=float(f), source=src, **base)
    print(f'{person}_{session} [{setup}, {len(grp)} clip]: side={"trái" if side == 0 else "phải"} facing={f:+.0f} '
          f'baseline({src}) ' + ' '.join(f'{k}={v:.1f}' for k, v in base.items()))
    for clip, r in raw.items():
        if clip == 'CALIBRATION' or clip not in TRAIN_CLIPS:
            continue
        cal = apply_calibration(r, base)
        labs = [rula_frame_label(row) for row in cal]
        lab = np.array([l for l, _ in labs], float); stab = np.array([s for _, s in labs], bool)
        cls = np.array([frame_class(row, l) for row, l in zip(cal, lab)])
        valid = np.isfinite(cal[:, :2]).all(1)
        judged = judged_mask(tt[clip], clip)
        xin = to_model_input(cal)
        for t in range(len(cal)):
            frames_rows.append(dict(person=person, session=session, clip=clip, t=round(float(tt[clip][t]), 2),
                                    **{k: cal[t, i] for i, k in enumerate(FEATURES)}, label=lab[t], cls=cls[t],
                                    stable=stab[t], judged=judged[t]))
        for s in range(0, len(cal) - WIN + 1, STRIDE):
            sl = slice(s, s + WIN)
            info = dict(person=person, session=session, clip=clip, start=round(float(tt[clip][s]), 2))
            if not judged[sl].all():
                INFO.append({**info, 'kept': False, 'reason': 'đoạn chuyển tư thế'}); continue
            l = lab[sl]; ok = np.isfinite(l)
            if valid[sl].mean() < MIN_VALID or ok.sum() == 0 or np.nanmean(cal[sl, iV]) < MIN_VIS_HIP:
                INFO.append({**info, 'kept': False, 'reason': 'thiếu điểm'}); continue
            pbad = np.nanmean(l)
            if MIN_PURE > pbad > 1 - MIN_PURE:
                INFO.append({**info, 'kept': False, 'reason': 'nhãn lẫn'}); continue
            if stab[sl][ok].mean() < MIN_STABLE:
                INFO.append({**info, 'kept': False, 'reason': 'sát ngưỡng'}); continue
            y = int(pbad >= 0.5)
            cs = cls[sl][cls[sl] >= 0]
            c = -1
            if len(cs):
                vals, cnt = np.unique(cs, return_counts=True)
                if cnt.max() / WIN >= CLS_MIN_PURE:
                    c = int(vals[cnt.argmax()])
            if (y == 0) != (c == 0):
                c = 0 if y == 0 else -1
            X.append(xin[sl]); Y.append(y); C.append(c); G.append(person)
            INFO.append({**info, 'kept': True, 'reason': '', 'label': y, 'cls': CLASSES[c] if c >= 0 else '-'})

X, Y, C, G = np.stack(X), np.array(Y), np.array(C), np.array(G)
np.savez_compressed('dataset_windows_v4.npz', X=X, Y=Y, C=C, G=G, classes=np.array(CLASSES))
info = pd.DataFrame(INFO); info.to_csv('windows_info_v4.csv', index=False)
pd.DataFrame(frames_rows).to_csv('frames_labeled_v4.csv', index=False)
json.dump(BASES, open('baselines_v4.json', 'w'), indent=2, ensure_ascii=False)

print(f'\nCửa sổ giữ lại: {len(Y)} | BAD: {int(Y.sum())} ({Y.mean():.0%}) | có lớp phụ: {(C >= 0).mean():.0%}')
print(info.assign(reason=info.reason.replace('', 'giữ lại')).groupby('reason').size().rename('số cửa sổ').to_string())
k = info[info.kept]
print('\nSố cửa sổ theo người × lớp phụ:')
print(pd.crosstab(k.person, k.cls).to_string())
print('\nKiểm teacher: % cửa sổ BAD và lớp phụ theo kịch bản quay (hàng = clip, cột = lớp teacher, %):')
ct = pd.crosstab(k['clip'], k['cls'], normalize='index').mul(100).round(0)
ct.insert(0, '%BAD', k.groupby('clip').label.mean().mul(100).round(0)); ct.insert(1, 'n', k.groupby('clip').size())
print(ct.to_string())
agree = k[k['clip'].isin(INTENT)].assign(intent=lambda d: d['clip'].map(INTENT))
print(f'\nTeacher trùng lớp dự kiến của kịch bản (khối A2–A3, B1–B9): {(agree.cls == agree.intent).mean():.0%} cửa sổ '
      '(clip quay "đạt" chưa chắc đúng 100%, con số này chỉ để phát hiện teacher sai lệch lớn).')

## Phần 5 — Train Transformer đa nhiệm (GOOD/BAD + 6 lớp), đánh giá LOSO, baseline Random Forest

- Kiến trúc giữ như v3 (d=64, 4 head, 2 encoder, ff=128, dropout 0,1, pre-norm, mean pooling) + thêm đầu `cls_head` 6 lớp.
- Loss = BCE(GOOD/BAD, pos_weight theo lớp) + `CLS_WEIGHT` × CrossEntropy(6 lớp, trọng số lớp, bỏ cửa sổ lớp −1).
- **LOSO**: train trên các người còn lại, test trên 1 người. Báo accuracy, κ, độ nhạy BAD, độ đặc hiệu GOOD, F1 BAD; lớp phụ: accuracy, macro-F1, ma trận nhầm.
- Model cuối train trên **toàn bộ** dữ liệu → `.pt` + `.onnx` + `posture_config.json`.

In [ ]:
import torch, torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix
from posture_features import CALIBRATED, ANGLE_SCALE, SHOULDER_TILT_ON, SHOULDER_TILT_OFF, MIN_SHOULDER_VIS

T, F = X.shape[1:]; K = len(CLASSES)
ANG_IDX = [i for i, k in enumerate(FEATURES) if not k.startswith('vis_')]
CAL_IDX = [FEATURES.index(k) for k in CALIBRATED]
EPOCHS, LR, BS, CLS_WEIGHT = 40, 2e-3, 128, 0.5
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'


class PostureTransformer(nn.Module):
    def __init__(self, f=F, t=T, n_cls=K, d=64, heads=4, layers=2, ff=128, drop=0.1):
        super().__init__()
        self.inp = nn.Linear(f, d)
        self.pos = nn.Parameter(torch.zeros(1, t, d))
        enc = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, 1)          # logit BAD (app dùng output này)
        self.cls_head = nn.Linear(d, n_cls)  # 6 lớp tư thế

    def forward(self, x):                    # x: (B,T,F) -> logit (B,), posture_logits (B,K)
        z = self.norm(self.enc(self.inp(x) + self.pos).mean(1))
        return self.head(z).squeeze(-1), self.cls_head(z)


def augment(xb):                             # giống v3: sai số đặt cam / hiệu chỉnh / nhiễu MediaPipe / mất khung
    xb = xb.clone(); B = xb.shape[0]; dev = xb.device
    xb[:, :, ANG_IDX] *= torch.empty(B, 1, 1, device=dev).uniform_(0.95, 1.05)
    xb[:, :, CAL_IDX] += torch.randn(B, 1, len(CAL_IDX), device=dev) * (2.0 / ANGLE_SCALE)
    xb[:, :, ANG_IDX] += torch.randn(B, T, len(ANG_IDX), device=dev) * (1.0 / ANGLE_SCALE)
    drop = torch.rand(B, T, 1, device=dev) < 0.05
    return torch.where(drop, torch.zeros_like(xb), xb)


def fit(Xtr, Ytr, Ctr, seed=SEED, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    m = PostureTransformer().to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS * (len(Xtr) // BS + 1))
    pos_w = torch.tensor((1 - Ytr.mean()) / max(Ytr.mean(), 1e-3), dtype=torch.float32, device=DEV)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    cnt = np.bincount(Ctr[Ctr >= 0], minlength=K).astype(float)
    cw = np.where(cnt > 0, cnt.sum() / (K * np.maximum(cnt, 1)), 0.0)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV), ignore_index=-1)
    Xt = torch.tensor(Xtr, device=DEV); Yt = torch.tensor(Ytr, dtype=torch.float32, device=DEV)
    Ct = torch.tensor(Ctr, dtype=torch.long, device=DEV)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(Xt), device=DEV); tot = 0
        for i in range(0, len(Xt), BS):
            idx = perm[i:i + BS]
            logit, cl = m(augment(Xt[idx]))
            loss = bce(logit, Yt[idx])
            if (Ct[idx] >= 0).any():
                loss = loss + CLS_WEIGHT * ce(cl, Ct[idx])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(idx)
        if verbose and (ep % 10 == 9 or ep == EPOCHS - 1):
            print(f'  epoch {ep + 1}: loss {tot / len(Xt):.3f}')
    return m.eval().cpu()


def predict(m, Xte):
    with torch.no_grad():
        logit, cl = m(torch.tensor(Xte))
    return torch.sigmoid(logit).numpy(), cl.argmax(1).numpy()


def scores(y, yh):
    y, yh = np.asarray(y).astype(int), np.asarray(yh).astype(int)
    tp = ((y == 1) & (yh == 1)).sum(); tn = ((y == 0) & (yh == 0)).sum()
    fp = ((y == 0) & (yh == 1)).sum(); fn = ((y == 1) & (yh == 0)).sum()
    acc = (tp + tn) / len(y)
    pe = (y == 1).mean() * (yh == 1).mean() + (y == 0).mean() * (yh == 0).mean()
    return dict(n=int(len(y)), acc=float(acc), kappa=float((acc - pe) / (1 - pe + 1e-9)), sens_bad=float(tp / max(tp + fn, 1)),
                spec_good=float(tn / max(tn + fp, 1)), f1_bad=float(2 * tp / max(2 * tp + fp + fn, 1)))


def cls_scores(c, ch):
    ok = c >= 0
    if not ok.any():
        return dict(n_cls=0, acc_cls=float('nan'), macro_f1=float('nan'))
    present = sorted(set(c[ok]))
    return dict(n_cls=int(ok.sum()), acc_cls=float((c[ok] == ch[ok]).mean()),
                macro_f1=float(f1_score(c[ok], ch[ok], labels=present, average='macro', zero_division=0)))


def win_stats(Xw):
    return np.concatenate([Xw.mean(1), Xw.std(1), Xw.min(1), Xw.max(1)], 1)


fmt = lambda r: ' '.join(f'{k}={v:.3f}' if isinstance(v, float) else f'{k}={v}' for k, v in r.items())
print(f'Dữ liệu: {len(Y)} cửa sổ, {Y.mean():.0%} BAD, đặc trưng {FEATURES}, thiết bị {DEV}\n')
res, res_rf, CM, CM_rf = {}, {}, np.zeros((K, K), int), np.zeros((K, K), int)
for p in np.unique(G):
    tr, te = G != p, G == p
    if len(np.unique(Y[tr])) < 2:
        print('Bỏ fold', p, '(train chỉ có 1 lớp)'); continue
    m = fit(X[tr], Y[tr], C[tr])
    pb, ch = predict(m, X[te])
    res[p] = {**scores(Y[te], pb >= 0.5), **cls_scores(C[te], ch)}
    ok = C[te] >= 0; CM += confusion_matrix(C[te][ok], ch[ok], labels=range(K))
    rf = RandomForestClassifier(300, class_weight='balanced', n_jobs=-1, random_state=0).fit(win_stats(X[tr]), Y[tr])
    trc = tr & (C >= 0)
    rfc = RandomForestClassifier(300, class_weight='balanced', n_jobs=-1, random_state=0).fit(win_stats(X[trc]), C[trc])
    chr_ = rfc.predict(win_stats(X[te]))
    res_rf[p] = {**scores(Y[te], rf.predict(win_stats(X[te]))), **cls_scores(C[te], chr_)}
    CM_rf += confusion_matrix(C[te][ok], chr_[ok], labels=range(K))
    print(f'LOSO {p} | Transformer : {fmt(res[p])}')
    print(f'LOSO {p} | RandomForest: {fmt(res_rf[p])}')
KEYS = ('acc', 'kappa', 'sens_bad', 'spec_good', 'f1_bad', 'acc_cls', 'macro_f1')
mean = lambda R: {k: float(np.nanmean([r[k] for r in R.values()])) for k in KEYS}
print('\nTrung bình LOSO | Transformer :', fmt(mean(res)))
print('Trung bình LOSO | RandomForest:', fmt(mean(res_rf)))
cm_df = pd.DataFrame(CM, index=[f'thật {c}' for c in CLASSES], columns=CLASSES)
print('\nMa trận nhầm lớp phụ (Transformer, gộp các fold LOSO):'); print(cm_df.to_string())
cm_df.to_csv('loso_confusion_v4.csv')

In [ ]:
print('Train model cuối trên toàn bộ dữ liệu ...')
model = fit(X, Y, C, verbose=True)
torch.save(model.state_dict(), f'{OUT_DIR}/posture_transformer.pt')
torch.onnx.export(model, torch.zeros(1, T, F), f'{OUT_DIR}/posture_transformer.onnx',
                  input_names=['x'], output_names=['logit', 'posture_logits'],
                  dynamic_axes={'x': {0: 'batch'}, 'logit': {0: 'batch'}, 'posture_logits': {0: 'batch'}},
                  opset_version=17, dynamo=False)
cfg = dict(version='v4', teacher='RULA nhóm B + LEAN_BACKWARD', side_cam_deg=90,
           features=FEATURES, calibrated=CALIBRATED, angle_scale=ANGLE_SCALE, window=int(T), fps=TARGET_FPS, threshold=0.5,
           outputs=['logit', 'posture_logits'], classes=CLASSES, class_rule=dict(flex_deg=FLEX, neck_ext_tol=NECK_EXT_TOL,
           lean_back_tol=LEAN_BACK_TOL, cls_min_pure=CLS_MIN_PURE),
           shoulder_tilt_on=SHOULDER_TILT_ON, shoulder_tilt_off=SHOULDER_TILT_OFF,
           shoulder_raise_on=None, shoulder_raise_off=None, shoulder_motion_std=None, min_shoulder_vis=MIN_SHOULDER_VIS,
           data=dict(source='video_tach (record_session.py, 11/2026)', people=sorted(map(str, np.unique(G))),
                     calib=f'CALIBRATION {CAL_START:.0f}–{CAL_START + CAL_SEC:.0f} s', margin_sec=MARGIN),
           n_train_windows=int(len(Y)), bad_ratio=float(Y.mean()),
           class_counts={c: int((C == i).sum()) for i, c in enumerate(CLASSES)},
           train=dict(epochs=EPOCHS, lr=LR, batch=BS, cls_weight=CLS_WEIGHT, seed=SEED),
           loso_transformer=res, loso_random_forest=res_rf, loso_mean_transformer=mean(res), loso_mean_random_forest=mean(res_rf),
           library=dict(mediapipe=mp.__version__, torch=torch.__version__, onnxruntime=onnxruntime.__version__))
json.dump(cfg, open(f'{OUT_DIR}/posture_config.json', 'w'), indent=2, ensure_ascii=False)
shutil.copy('posture_features.py', f'{OUT_DIR}/posture_features.py')
print('Đã lưu posture_transformer.pt / .onnx / posture_config.json')

## Phần 6 — Kiểm ONNX (khớp PyTorch, chạy đúng như `engine.py`) + so với model v3 (nếu có)

`engine.py` gọi `sess.run(None, {"x": buf[None]})[0][0]` rồi lấy sigmoid. Ô này gọi y hệt để chắc app dùng được file mới.
Nếu Input có `posture_transformer.onnx` của v3 (khác thư mục Output), so v3 với v4 trên toàn bộ cửa sổ mới.
Lưu ý: v4 được train trên chính các cửa sổ này nên số của v4 ở đây là số **trên tập train**; số công bằng của v4 là LOSO ở Phần 5.
Với v3, đây là dữ liệu chưa từng thấy (người/buổi quay mới).

In [ ]:
import onnxruntime as ort
s = ort.InferenceSession(f'{OUT_DIR}/posture_transformer.onnx', providers=['CPUExecutionProvider'])
print('ONNX outputs:', [(o.name, o.shape) for o in s.get_outputs()])
idx = np.r_[np.where(Y == 0)[0][:4], np.where(Y == 1)[0][:4]]
lo, cl = s.run(None, {'x': X[idx]})
with torch.no_grad():
    lt, ct = model(torch.tensor(X[idx]))
assert np.allclose(1 / (1 + np.exp(-lo)), torch.sigmoid(lt).numpy(), atol=1e-4) and np.allclose(cl, ct.numpy(), atol=1e-3)
logit = s.run(None, {'x': X[idx[0]][None]})[0][0]          # đúng cách engine.py gọi
assert np.ndim(logit) == 0, 'engine.py cần output đầu tiên là logit vô hướng'
print('✅ ONNX khớp PyTorch; output[0][0] là logit vô hướng như engine.py cần')
print('P(BAD):', (1 / (1 + np.exp(-lo))).round(3), '| nhãn:', Y[idx])
print('Lớp:', [CLASSES[i] for i in cl.argmax(1)])

def _is_v3(p):                                # bỏ qua ONNX v4 từ Output lần trước (cùng tên file)
    c = Path(p).with_name('posture_config.json')
    try:
        return not c.exists() or json.load(open(c, encoding='utf-8')).get('version') != 'v4'
    except Exception:
        return True
old = [str(p) for p in Path(INPUT_ROOT).rglob('posture_transformer.onnx') if _is_v3(p)] if Path(INPUT_ROOT).exists() else []
if old:
    so = ort.InferenceSession(old[0], providers=['CPUExecutionProvider'])
    pv3 = 1 / (1 + np.exp(-so.run(None, {'x': X})[0]))
    print(f'\nModel v3 ({old[0]}) trên dữ liệu mới:')
    cmp = {p: scores(Y[G == p], pv3[G == p] >= 0.5) for p in np.unique(G)}
    for p, r in cmp.items():
        print(f'  {p}: {fmt(r)}   | v4 LOSO: {fmt({k: res[p][k] for k in ("acc", "kappa", "sens_bad", "spec_good")}) if p in res else "-"}')
    cfg['v3_on_new_data'] = cmp
    json.dump(cfg, open(f'{OUT_DIR}/posture_config.json', 'w'), indent=2, ensure_ascii=False)
else:
    print('\n(Không có posture_transformer.onnx v3 trong Input — bỏ qua so sánh v3.)')

## Phần 7 — Cam trước: lệch vai + nhún vai (không train, gợi ý ngưỡng)

- Baseline theo người = trung vị giây 3–18 của `CALIBRATION_FRONT`.
- `tilt_rel` = lệch vai − baseline (độ, lấy trị tuyệt đối); `raise_rel` = (baseline − `shoulder_raise`) / baseline, dạng phân số như `engine.py` (dương = vai nâng lên, vd 0,08 = 8%).
  Cả hai làm mượt median 5 khung như app.
- Nhóm khung: **GOOD** (GOOD_01/02, ngồi đúng = âm tính), **nhẹ** (giây 3–15) và **rõ** (giây 18–27) của SHOULDER_TILT / SHOULDER_RAISE / SHRUG.
- Ngưỡng gợi ý `on` = giữa P99 của GOOD và P10 của mức **nhẹ** (để bắt cả mức nhẹ); nếu hai nhóm chồng nhau thì dùng Youden J giữa GOOD và mức **rõ**. `off` = 0,7 × `on`. Chỉ là gợi ý từ vài người, ghi vào `shoulder_suggest_v4.json`,
  **không** tự ghi vào `posture_config.json` (app đang lấy ngưỡng vai từ file này khi `app_config.json` để `null`, nên v4 giữ nguyên 3,5°/2,5° và nhún vai tắt như v3).

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from posture_features import sequence_shoulder_tilt, shoulder_raise

def med5(a):
    return pd.Series(a).rolling(5, center=True, min_periods=1).median().to_numpy()

front = {}
for s, f in LM.items():
    p, ses, clip, cam = parse(s)
    if cam == 'FRONT':
        front.setdefault(f'{p}_{ses}', {})[clip] = np.load(f)
rows = []
for ps, clips in sorted(front.items()):
    if 'CALIBRATION' not in clips:
        print(ps, ': thiếu CALIBRATION_FRONT -> bỏ'); continue
    z = clips['CALIBRATION']; t = z['t'] - z['t'][0]; sel = (t >= CAL_START) & (t < CAL_START + CAL_SEC)
    b_tilt = np.nanmedian(sequence_shoulder_tilt(z['lm'])[sel])
    b_raise = np.nanmedian(np.array([shoulder_raise(p) for p in z['lm']])[sel])
    for clip, z in clips.items():
        if clip == 'CALIBRATION':
            continue
        t = z['t'] - z['t'][0]
        tilt = med5(np.abs(sequence_shoulder_tilt(z['lm']) - b_tilt))
        rz = med5((b_raise - np.array([shoulder_raise(p) for p in z['lm']])) / b_raise)   # đúng công thức engine.py
        for i in range(len(t)):
            if clip.startswith('GOOD'):
                grp = 'GOOD' if MARGIN <= t[i] < SCRIPT[clip][0] - MARGIN else None
            else:
                grp = 'nhẹ' if MARGIN <= t[i] < 15 else 'rõ' if 15 + MARGIN <= t[i] < 30 - MARGIN else None
            if grp:
                rows.append(dict(person=ps, clip=clip, group=grp, t=t[i], tilt_rel=tilt[i], raise_rel=rz[i]))
fr = pd.DataFrame(rows)

def suggest(neg, mild, clear):
    # Ưu tiên bắt được mức NHẸ: ngưỡng = giữa P99 của GOOD và P10 của mức nhẹ. Hai nhóm chồng nhau thì dùng Youden J
    # giữa GOOD và mức rõ (lấy giữa khoảng ngưỡng cùng đạt J lớn nhất).
    neg, mild, clear = (np.asarray(v)[np.isfinite(v)] for v in (neg, mild, clear))
    if not len(neg) or not len(clear):
        return None
    g99 = float(np.percentile(neg, 99)); m10 = float(np.percentile(mild, 10)) if len(mild) else -np.inf
    if m10 > g99:
        thr, rule = (g99 + m10) / 2, 'giữa P99 GOOD và P10 mức nhẹ'
    else:
        cand = np.unique(np.round(np.r_[neg, clear], 2))
        J = np.array([(clear > c).mean() - (neg > c).mean() for c in cand])
        top = cand[J >= J.max() - 0.01]
        thr, rule = (top.min() + top.max()) / 2, 'Youden GOOD vs mức rõ'
    thr = float(np.round(thr, 3))
    return dict(on=thr, off=float(np.round(thr * 0.7, 3)), rule=rule, good_p99=g99, mild_p10=m10,
                tpr_clear=float((clear > thr).mean()), tpr_mild=float((mild > thr).mean()) if len(mild) else None,
                fpr_good=float((neg > thr).mean()))

SUG = {}
if len(fr):
    def kind(c):
        return 'tilt' if c.startswith('SHOULDER_TILT') else 'raise1' if c.startswith('SHOULDER_RAISE') else 'shrug' if 'SHRUG' in c else 'good'
    fr['kind'] = fr['clip'].map(kind)
    print('Trung vị / P90 theo nhóm (độ cho tilt_rel, phân số cho raise_rel):')
    print(fr.groupby(['kind', 'group'])[['tilt_rel', 'raise_rel']].quantile([0.5, 0.9]).unstack().round(2).to_string())
    neg = fr[fr.kind == 'good']
    for k, kinds in (('tilt', ['tilt']), ('raise', ['shrug', 'raise1'])):
        d = fr[fr.kind.isin(kinds)]
        v = SUG[k] = suggest(neg[f'{k}_rel'], d[d.group == 'nhẹ'][f'{k}_rel'], d[d.group == 'rõ'][f'{k}_rel'])
        if v:
            cur = f' (app đang dùng {SHOULDER_TILT_ON}/{SHOULDER_TILT_OFF}°)' if k == 'tilt' else ''
            print(f'\n{k}: gợi ý on={v["on"]} off={v["off"]} [{v["rule"]}] → bắt {v["tpr_clear"]:.0%} khung mức rõ, '
                  f'{v["tpr_mild"] or 0:.0%} khung mức nhẹ, báo nhầm {v["fpr_good"]:.0%} khung GOOD{cur}')
    json.dump(dict(note='Gợi ý từ clip cam trước, chưa áp dụng vào app', people=sorted(fr.person.unique()), **SUG),
              open(f'{OUT_DIR}/shoulder_suggest_v4.json', 'w'), indent=2, ensure_ascii=False)
    fr.to_csv('front_shoulder_v4.csv', index=False)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a, col, unit in ((ax[0], 'tilt_rel', 'độ'), (ax[1], 'raise_rel', 'phân số')):
        for (kd, g), d in fr.groupby(['kind', 'group']):
            a.hist(d[col].dropna(), bins=40, alpha=0.5, label=f'{kd}/{g}', density=True)
        key = 'tilt' if col == 'tilt_rel' else 'raise'
        if SUG.get(key):
            a.axvline(SUG[key]['on'], c='r', ls='--', label='gợi ý')
        a.set_title(col); a.set_xlabel(unit); a.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/shoulder_check_v4.png', dpi=110); plt.show()
else:
    print('Không có clip cam trước — bỏ qua.')

## Phần 8 — Đóng gói Output

In [ ]:
for f in ('windows_info_v4.csv', 'loso_confusion_v4.csv', 'baselines_v4.json'):
    if os.path.exists(f):
        shutil.copy(f, OUT_DIR)
shutil.make_archive(f'{WORK}/posture_model_v4', 'zip', OUT_DIR)
print('Output:', sorted(os.listdir(OUT_DIR)))
print('→ Tải posture_model_v4.zip. Landmark cache ở', LM_DIR, '(thêm Output này làm Input lần sau để khỏi trích lại).')
print('\nTóm tắt LOSO Transformer:', fmt(mean(res)))

## Đưa model vào app

1. Sao lưu `GIT_KLTN/posture_model/posture_transformer.onnx`, `.pt`, `posture_config.json` hiện tại (v3).
2. Chép 3 file cùng tên từ `posture_model_v4.zip` đè vào `posture_model/`. **Không** chép đè `posture_features.py` (file trong zip là bản sao để đối chiếu).
3. Chạy app trên video quay sẵn để kiểm: `python realtime_app.py --side-video ..._SIDE.mp4 --front-video ..._FRONT.mp4` (dùng clip TRANSITION_01).
4. Ngưỡng vai trong `shoulder_suggest_v4.json` chỉ là gợi ý: nếu muốn dùng, sửa `shoulder_tilt_on/off`, `shoulder_raise_on/off` trong `app_config.json`.